In [2]:
import os
from datetime import datetime
import matplotlib.pyplot as plt
import shutil
import random
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import cv2

# 安装 ultralytics
!pip install ultralytics -q

from ultralytics import YOLO
from IPython.display import display, Image

# 设置随机种子
random.seed(42)
np.random.seed(42)

print("="*60)
print("Mosquito Binary Classifier - Training Pipeline")
print("="*60)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mosquito Binary Classifier - Training Pipeline


In [3]:
print("="*60)
print("/kaggle/input 下的目录结构（三级）:")
print("="*60)

for item in os.listdir('/kaggle/input'):
    item_path = os.path.join('/kaggle/input', item)
    if os.path.isdir(item_path):
        print(f"📁 {item}")
        
        # 二级目录
        try:
            sub_items = os.listdir(item_path)
            for sub in sub_items[:5]:  # 显示前5个
                sub_path = os.path.join(item_path, sub)
                if os.path.isdir(sub_path):
                    print(f"   └── 📁 {sub}")
                    
                    # 三级目录
                    try:
                        third_items = os.listdir(sub_path)
                        for third in third_items[:3]:  # 显示前3个
                            third_path = os.path.join(sub_path, third)
                            if os.path.isdir(third_path):
                                print(f"        └── 📁 {third}")
                            else:
                                print(f"        └── 📄 {third}")
                    except:
                        pass
                else:
                    print(f"   └── 📄 {sub}")
        except:
            pass
        print()

/kaggle/input 下的目录结构（三级）:
📁 datasets
   └── 📁 viviorangeani
        └── 📁 mosquito
   └── 📁 vishakkbhat
        └── 📁 mosquito-data



In [4]:
# 1. 设置路径
# ============================================
IMAGE_DIR = "/kaggle/input/datasets/vishakkbhat/mosquito-data/train_images/"
LABEL_DIR = "/kaggle/input/datasets/viviorangeani/mosquito/labels/"
DATA_YAML_PATH = "/kaggle/input/datasets/viviorangeani/mosquito/data.yaml"

# 工作目录
WORK_DIR = "/kaggle/working/yolo_dataset"
os.makedirs(WORK_DIR, exist_ok=True)

print(f"\n图像目录: {IMAGE_DIR}")
print(f"标签目录: {LABEL_DIR}")
print(f"工作目录: {WORK_DIR}")


图像目录: /kaggle/input/datasets/vishakkbhat/mosquito-data/train_images/
标签目录: /kaggle/input/datasets/viviorangeani/mosquito/labels/
工作目录: /kaggle/working/yolo_dataset


In [5]:
# 2. 获取所有匹配的图像和标签文件
# ============================================
print("\n正在匹配图像和标签文件...")

# 获取所有标签文件（.txt）
label_files = [f for f in os.listdir(LABEL_DIR) if f.endswith('.txt')]
print(f"找到 {len(label_files)} 个标签文件")

# 匹配对应的图像文件
matched_pairs = []
for label_file in tqdm(label_files, desc="匹配图像"):
    uuid = label_file.replace('.txt', '')
    
    # 检查图像是否存在（支持 .jpeg 和 .jpg）
    img_path_jpeg = os.path.join(IMAGE_DIR, f"{uuid}.jpeg")
    img_path_jpg = os.path.join(IMAGE_DIR, f"{uuid}.jpg")
    
    if os.path.exists(img_path_jpeg):
        img_path = img_path_jpeg
        matched_pairs.append((img_path, os.path.join(LABEL_DIR, label_file)))
    elif os.path.exists(img_path_jpg):
        img_path = img_path_jpg
        matched_pairs.append((img_path, os.path.join(LABEL_DIR, label_file)))
    else:
        print(f"警告: 找不到图像 {uuid}.jpeg/jpg，跳过")

print(f"成功匹配 {len(matched_pairs)} 对图像-标签")

if len(matched_pairs) == 0:
    raise ValueError("没有找到匹配的图像-标签对！")


正在匹配图像和标签文件...
找到 8025 个标签文件


匹配图像: 100%|██████████| 8025/8025 [00:16<00:00, 473.34it/s]

成功匹配 8025 对图像-标签


In [6]:
# ============================================
# 3. 划分数据集 (80% 训练, 20% 测试)
# ============================================
print("\n划分数据集 (80% 训练, 20% 测试)...")

# 获取所有唯一标识符
all_uuids = [os.path.basename(img_path).replace('.jpeg', '').replace('.jpg', '') 
             for img_path, _ in matched_pairs]

# 划分
train_uuids, test_uuids = train_test_split(
    all_uuids, 
    test_size=0.2, 
    random_state=42,
    stratify=None  # 二分类比例已经平衡
)

print(f"训练集: {len(train_uuids)} 张图像")
print(f"测试集: {len(test_uuids)} 张图像")

# 创建 UUID 到路径的映射
img_path_map = {os.path.basename(p).replace('.jpeg', '').replace('.jpg', ''): p 
                for p, _ in matched_pairs}
label_path_map = {os.path.basename(l).replace('.txt', ''): l 
                  for _, l in matched_pairs}


划分数据集 (80% 训练, 20% 测试)...
训练集: 6420 张图像
测试集: 1605 张图像


In [7]:
# 4. 创建 YOLO 标准目录结构
# ============================================
print("\n创建 YOLO 标准目录结构...")

# 创建目录
for split in ['train', 'test']:
    os.makedirs(os.path.join(WORK_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(WORK_DIR, split, 'labels'), exist_ok=True)

# 复制/创建软链接（Kaggle 环境下使用复制）
def copy_files(uuids, split_name):
    """复制图像和标签文件到目标目录"""
    for uuid in tqdm(uuids, desc=f"复制 {split_name} 集"):
        # 复制图像
        src_img = img_path_map.get(uuid)
        if src_img and os.path.exists(src_img):
            dst_img = os.path.join(WORK_DIR, split_name, 'images', os.path.basename(src_img))
            shutil.copy2(src_img, dst_img)
        
        # 复制标签
        src_label = label_path_map.get(uuid)
        if src_label and os.path.exists(src_label):
            dst_label = os.path.join(WORK_DIR, split_name, 'labels', 
                                    os.path.basename(src_label).replace('.txt', '.txt'))
            shutil.copy2(src_label, dst_label)

copy_files(train_uuids, 'train')
copy_files(test_uuids, 'test')

print(f"\n目录结构创建完成:")
print(f"训练图像: {len(os.listdir(os.path.join(WORK_DIR, 'train', 'images')))}")
print(f"训练标签: {len(os.listdir(os.path.join(WORK_DIR, 'train', 'labels')))}")
print(f"测试图像: {len(os.listdir(os.path.join(WORK_DIR, 'test', 'images')))}")
print(f"测试标签: {len(os.listdir(os.path.join(WORK_DIR, 'test', 'labels')))}")


创建 YOLO 标准目录结构...


复制 test 集: 100%|██████████| 1605/1605 [00:42<00:00, 37.85it/s]


目录结构创建完成:
训练图像: 6420
训练标签: 6420
测试图像: 1605
测试标签: 1605


In [8]:
# 5. 创建 data.yaml 配置文件
# ============================================
print("\n创建 data.yaml 配置文件...")

yaml_content = f"""
# Mosquito Binary Classification Dataset
# Aedes albopictus (class 0) vs Other Mosquitoes (class 1)

path: {WORK_DIR}
train: train/images
val: test/images 
test: test/images

nc: 2
names:
  0: albopictus
  1: Other_mosquito
"""

yaml_path = os.path.join(WORK_DIR, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"data.yaml 已保存到: {yaml_path}")
print(yaml_content)


创建 data.yaml 配置文件...
data.yaml 已保存到: /kaggle/working/yolo_dataset/data.yaml

# Mosquito Binary Classification Dataset
# Aedes albopictus (class 0) vs Other Mosquitoes (class 1)

path: /kaggle/working/yolo_dataset
train: train/images
val: test/images 
test: test/images

nc: 2
names:
  0: albopictus
  1: Other_mosquito



In [9]:
# 6. 验证标签格式
# ============================================
print("\n验证标签格式...")
sample_label_dir = os.path.join(WORK_DIR, 'train', 'labels')
sample_labels = os.listdir(sample_label_dir)[:3]

for label_file in sample_labels:
    with open(os.path.join(sample_label_dir, label_file), 'r') as f:
        content = f.read().strip()
    print(f"{label_file}: {content}")


验证标签格式...
fdd4895f-e3b2-4190-8326-006ba90488e2.txt: 0 0.483073 0.442383 0.184896 0.183594
7e95ed5f-eaa5-416f-be7f-71621e011439.txt: 1 0.454200 0.626488 0.851521 0.574901
5bf8ad4a-a7c9-400d-8245-01abd13610b5.txt: 1 0.577606 0.595443 0.314433 0.173689


In [ ]:
# 7. 训练 YOLOv8n 模型
# ============================================
print("\n" + "="*60)
print("开始训练 YOLOv8n 模型")
print("="*60)

# 加载模型
model = YOLO('yolov8n.pt')

# 训练参数
results = model.train(
    data=yaml_path,
    epochs=25,
    imgsz=640,
    batch=16,              
    device=0,              # 使用 GPU
    workers=4,             # 数据加载线程
    fraction=0.3,
    patience=10,           # 早停耐心值
    save=True,             # 保存最佳模型
    save_period=5,         # 每 5 轮保存一次
    project='mosquito_detector',
    name='yolov8n_aedes_binary',
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    seed=42,
    verbose=True,
    plots=True             # 生成训练图表
)

print("\n训练完成！")


开始训练 YOLOv8n 模型
Ultralytics 8.4.53 🚀 Python-3.10.10 torch-2.0.0 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_aedes_binary, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

In [ ]:
# 8. 模型验证
# ============================================
print("\n" + "="*60)
print("模型验证")
print("="*60)

# 在测试集上验证
val_results = model.val(
    data=yaml_path,
    split='test',
    conf=0.25,
    iou=0.45
)

print(f"\n验证结果:")
print(f"mAP50: {val_results.box.map50:.4f}")
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall: {val_results.box.mr:.4f}")

In [ ]:
# 9. 测试集推理可视化
# ============================================
print("\n" + "="*60)
print("测试集推理（显示前 5 张）")
print("="*60)

# 选择测试集前 5 张图像
test_images_dir = os.path.join(WORK_DIR, 'test', 'images')
test_images = os.listdir(test_images_dir)[:5]

for img_name in test_images:
    img_path = os.path.join(test_images_dir, img_name)
    
    # 推理
    results = model.predict(
        source=img_path,
        conf=0.25,
        save=True,
        project='mosquito_detector',
        name='test_predictions',
        exist_ok=True,
        show=False,
        verbose=False
    )

print("\n预测结果已保存到: mosquito_detector/test_predictions/")

In [ ]:
# 10. 显示训练结果图表
# ============================================
print("\n" + "="*60)
print("训练结果可视化")
print("="*60)

result_images = [
    'mosquito_detector/yolov8n_aedes_binary/confusion_matrix.png',
    'mosquito_detector/yolov8n_aedes_binary/results.png',
    'mosquito_detector/yolov8n_aedes_binary/train_batch0.jpg',
    'mosquito_detector/yolov8n_aedes_binary/val_batch0_labels.jpg'
]

for img_path in result_images:
    if os.path.exists(img_path):
        print(f"\n显示: {img_path}")
        display(Image(filename=img_path, width=800))

# ============================================
# 11. 保存最佳模型
# ============================================
best_model_path = 'mosquito_detector/yolov8n_aedes_binary/weights/best.pt'
if os.path.exists(best_model_path):
    print(f"\n✅ 最佳模型已保存: {best_model_path}")
    
    # 复制到工作目录根目录方便下载
    shutil.copy(best_model_path, '/kaggle/working/best_model.pt')
    print("✅ 模型已复制到: /kaggle/working/best_model.pt")

print("\n" + "="*60)
print("训练完成！")
print("="*60)

In [ ]:

# For all images in the Train dataset...
#
#
imagePath='/kaggle/input/mosquito-data/train_images/'
fig = plt.figure()

for idx, data in trainDF.iterrows():
        
    if idx > 3:    # for now 
        break
        
    filePath = imagePath + data.img_fName
    print( idx, filePath, data.class_label )

    img = cv2.imread( filePath, cv2.IMREAD_COLOR )

    print('Image shape (y,x) = ', img.shape )
    
    box = int(idx + 1)
    ax1 = fig.add_subplot( 2, 2, box )
    ax1.imshow(img)
        
        
plt.show()


In [ ]:
#  Use the example file as a model and
#     just set random classifications for each image
#        See the file under the 'Outputs' tab for the Notebook

prepareSubmissionFile()



In [ ]:
# Mission Complete!
##################################################################################
global_later = datetime.now()
print("#####  MosquitoAlert-Baseline    - Total EXECUTION Time =", (global_later - global_now), global_later )

